# 2. Train a model

Training needs a folder with **one sub-folder of `.tif` crops per class**, the layout `preprocess.ipynb`
produces. Here we use the class folders at the top of `/data/Hier-Zooplankton`, the training data of
the released models.

To keep the example quick, this run uses at most 300 images per class and 3 epochs, which takes a few
minutes on a GPU. The released models used up to 12,000 images per class and 40 epochs; the config to
reproduce them is `configs/mnr_swin_t_cnn_noise.yaml` in the repository.

Requires `pip install hierroute jupyter`. The example data lives at `/data/Hier-Zooplankton` on the lab server; change `DATA_DIR` below if your copy is elsewhere. Everything this notebook writes goes under `WORK_DIR`.

In [ ]:
from pathlib import Path

DATA_DIR = Path("/data/Hier-Zooplankton")        # example dataset (internal)
WORK_DIR = Path.home() / "hierroute_example"      # everything the example notebooks write goes here
WORK_DIR.mkdir(parents=True, exist_ok=True)

## The taxonomy and how folders map onto it

A **hierarchy** defines the classes and how they nest. The released models use the bundled
`mnr_17leaf` hierarchy:
- every leaf class (one with no children) is read from a folder of the same name
- `folder_map` pools several folders into one class (e.g. `Bosminidae`, `Eubosmina` → `Bosmina`)

To use your own taxonomy, write a YAML file in the same format (see the README) and pass its path
instead of `"mnr_17leaf"`.

In [ ]:
from hierroute.config import load_hierarchy

spec = load_hierarchy("mnr_17leaf")

def show(node, depth=0):
    print("    " * depth + node)
    for child in spec.tree[node]:
        show(child, depth + 1)

show("root")
print("\nfolder_map:")
for cls, folders in spec.folder_map.items():
    print(f"  {cls:18s} <- {folders}")

In [ ]:
# Images available for each class (top-level folders only; the config also reads data.subdirectories)
import pandas as pd

def n_images(cls):
    return sum(len(list((DATA_DIR / f).glob("*.tif"))) for f in spec.folder_map.get(cls, [cls])
               if (DATA_DIR / f).is_dir())

pd.Series({cls: n_images(cls) for cls in spec.leaves}, name="images").sort_values()

## Configure the run

`load_train_config` starts from the defaults (`hierroute/config.py`, `TRAIN_DEFAULTS`), applies an
optional YAML file, then your overrides. All settings are documented in the README.

| Setting | Meaning |
|---|---|
| `model.backbone` | `swin_t`, `swin_s`, `efficientnet_b0` |
| `model.expert` | `cnn`, `mlp`, `linear` |
| `augment.background_noise` | add synthetic specks and fibres to the background (the `noise` model) |
| `data.max_class_size` | cap per class |
| `data.exclude_log` | images held out for the messy-background test set |

In [ ]:
import yaml
from hierroute.config import load_train_config

config = load_train_config(overrides={
    "hierarchy": "mnr_17leaf",
    "model": {"backbone": "swin_t", "expert": "cnn"},
    "data": {
        "data_dir": str(DATA_DIR),
        "subdirectories": ["IssacData"],
        "exclude_log": "Messy_background/moved_images/log.md",
        "max_class_size": 300,          # released models: 12000
    },
    "augment": {"background_noise": True},
    "train": {"max_epochs": 3, "batch_size": 64, "num_workers": 8},   # released models: 40 epochs
    "out_dir": str(WORK_DIR / "training_result"),
    "run_name": "demo_swin_t_cnn_noise",
})
print(yaml.safe_dump(config, sort_keys=False))

## Train

This builds the dataset, splits it 70 / 10 / 20 into train / validation / test (stratified, seed
666), and trains. It keeps the checkpoint with the best validation accuracy, then scores the
held-out test split with it.

The run folder is returned and contains:
- `best_model.pt`, the checkpoint
- `model_config.json`, so it loads like a released model
- `train_config.yaml`, `training_info.json`, `eval_results.json`, `predictions.npz`

An existing run folder is never overwritten; a `_1`, `_2`, … suffix is added instead.

In [ ]:
from hierroute.train import train

run_dir = train(config)
run_dir

## Results

In [ ]:
from hierroute import Visualize

vis = Visualize(run_dir)
vis.plot_train()          # loss / accuracy / F1 per epoch

In [ ]:
import json

levels = json.loads((run_dir / "eval_results.json").read_text())["level_results"]
pd.DataFrame({f"level {d}": r["overall"] for d, r in levels.items()}).T

## Next

Use the trained model exactly like a released one, by passing its run folder as `weights` (see
`inference.ipynb`):

```bash
hierroute evaluate ~/hierroute_example/erie_crops --labelled \
    --weights ~/hierroute_example/training_result/demo_swin_t_cnn_noise --out results/
```

The same training from the terminal:

```bash
hierroute train --config configs/mnr_swin_t_cnn_noise.yaml --data-dir /data/Hier-Zooplankton \
    --set data.max_class_size=300 --epochs 3 --run-name demo_swin_t_cnn_noise
```